# Кейс 1. First Blood и вероятность победы

Цель: проверить, чаще ли побеждает команда первого убийства и ослабевает ли эффект после пяти минут. Полный путь: источник → учебный RAW → сверка → STAGE → MART → качество и Iceberg → вывод.

Учебные таблицы имеют префикс `practice_case_01_`; production-таблицы не изменяются. Запускайте ячейки сверху вниз.

## Методика и метрики

Это описательный статистический анализ, а не причинная модель. Для каждого матча определяется первое корректно зафиксированное убийство, затем команда, совершившая это убийство, сравнивается с командой-победителем. После этого матчи группируются по времени первого убийства.

Основная метрика — `first_blood_win_rate`: доля матчей, в которых команда, совершившая первое убийство, в итоге выиграла. `lift` — отклонение этой доли от базового значения 50 процентов. Дополнительно используются число матчей в группе и среднее время первого убийства. Для небольших групп результаты нужно интерпретировать осторожно: доверительные интервалы и статистическая проверка в этом notebook не рассчитываются.

## 1. Источники и детализация данных

Используются PostgreSQL `dota.matches`, `dota.match_players` и MongoDB `dota_events.kill_events`. Ключ между системами — `match_id`; убийца связывается по `steam_account_id = account_id`. Пароли и URI читаются только из environment variables и не выводятся.

In [ ]:
import os
import psycopg2
from pymongo import MongoClient

pg_dsn = os.getenv("SOURCE_POSTGRES_DSN")
mongo_uri = os.getenv("SOURCE_MONGODB_URI")
if not pg_dsn or not mongo_uri:
    raise RuntimeError("Нужны SOURCE_POSTGRES_DSN и SOURCE_MONGODB_URI")

with psycopg2.connect(pg_dsn) as connection:
    connection.set_session(readonly=True, autocommit=True)
    with connection.cursor() as cursor:
        for table in ["matches", "match_players"]:
            cursor.execute("select count(*) from dota." + table)
            print(f"PostgreSQL {table}:", cursor.fetchone()[0])

with MongoClient(mongo_uri, serverSelectionTimeoutMS=5000) as client:
    collection = client["dota_events"]["kill_events"]
    print("MongoDB kill_events:", collection.count_documents({}))
    print("Пример полей:", sorted(collection.find_one({}, {"_id": 0}).keys()))

## 2. Построение учебного RAW

Spark читает источники напрямую через JDBC и MongoDB connector. RAW сохраняет поля источника без бизнес-агрегаций; добавляются только `src_modified_stamp` и `sys_insert_stamp`. `createOrReplace()` делает повторный учебный запуск воспроизводимым.

In [ ]:
from psycopg2.extensions import parse_dsn
from pyspark.sql import functions as F, types as T
from lakehouse_spark import SparkSessionContext
from lakehouse_publish import publish_table

pg = parse_dsn(pg_dsn)
jdbc_url = f"jdbc:postgresql://{pg['host']}:{pg.get('port', '5432')}/{pg['dbname']}"
jdbc_options = {"user": pg["user"], "password": pg["password"], "driver": "org.postgresql.Driver"}
kill_schema = T.StructType([
    T.StructField("_id", T.StringType()), 
    T.StructField("match_id", T.LongType()),
    T.StructField("steam_account_id", T.LongType()), 
    T.StructField("created_at", T.TimestampType()),
    T.StructField("time", T.IntegerType()), 
    T.StructField("attacker", T.IntegerType()),
    T.StructField("target", T.IntegerType()),
])

with SparkSessionContext("practice-case-01-raw") as ctx:
    spark = ctx
    
    matches = (
        spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", "dota.matches")
            .options(**jdbc_options)
            .load()
            .select(
                "match_id", "start_time", "duration", "radiant_win", 
                "first_blood_time", "stratz_parsed_at"
            )
            .withColumnRenamed("stratz_parsed_at", "src_modified_stamp")
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )
    
    players = (
        spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", "dota.match_players")
            .options(**jdbc_options)
            .load()
            .select(
                "match_id", "player_slot", "account_id", 
                "hero_id", "win", "loaded_at"
            )
            .withColumnRenamed("loaded_at", "src_modified_stamp")
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )
    
    kills = (
        spark.read.format("mongodb")
            .option("connection.uri", mongo_uri)
            .option("database", "dota_events")
            .option("collection", "kill_events")
            .schema(kill_schema)
            .load()
            .select(
                F.col("_id").cast("string").alias("id"), 
                "match_id", "steam_account_id", "time", "attacker", "target", 
                F.col("created_at").alias("src_modified_stamp")
            )
            .withColumn("sys_insert_stamp", F.current_timestamp())
    )
    
    for name, frame in {"matches": matches, "match_players": players, "kill_events": kills}.items():
        output = f"lakehouse.raw.practice_case_01_{name}"
        with ctx.lineage(
            f"practice-case-01-raw-{name}", 
            output=f"raw.practice_case_01_{name}", 
            description="Загрузка исходных данных для кейса 1",
            dataset_descriptions={f"raw.practice_case_01_{name}": f"Учебная RAW-таблица {name} для анализа первого убийства."}
        ):
            frame.writeTo(output).using("iceberg").createOrReplace()
            publish_table("raw", f"practice_case_01_{name}", frame, description=f"Учебная RAW-таблица {name} для анализа первого убийства")
        print(output, frame.count())

## 3. Сверка RAW с источником

Сравниваем количество строк, уникальные ключи и диапазоны времени. Любое расхождение нужно объяснить до построения STAGE.

In [ ]:
with psycopg2.connect(pg_dsn) as connection:
    connection.set_session(readonly=True, autocommit=True)
    
    with connection.cursor() as cursor:
        cursor.execute("""
            select count(*), count(distinct match_id), min(start_time), max(start_time) 
            from dota.matches
        """)
        
        source_matches = cursor.fetchone()
        cursor.execute("""
            select count(*), count(distinct (match_id, player_slot)) 
            from dota.match_players
        """)
        
        source_players = cursor.fetchone()
        
with MongoClient(mongo_uri, serverSelectionTimeoutMS=5000) as client:
    source_kills = client["dota_events"]["kill_events"].count_documents({})

with SparkSessionContext("practice-case-01-reconcile") as spark:
    raw_matches = spark.sql("""
        select count(*), count(distinct match_id), min(start_time), max(start_time) 
        from lakehouse.raw.practice_case_01_matches
    """).first()
    
    raw_players = spark.sql("""
        select count(*), count(distinct struct(match_id, player_slot)) 
        from lakehouse.raw.practice_case_01_match_players
    """).first()
    
    raw_kills = spark.table("lakehouse.raw.practice_case_01_kill_events").count()
    
    checks = {
        "matches": (source_matches, tuple(raw_matches)), 
        "players": (source_players, tuple(raw_players)), 
        "kills": (source_kills, raw_kills)
    }
    for name, (source_value, raw_value) in checks.items():
        print(name, "PASS" if source_value == raw_value else "FAIL", {"source": source_value, "raw": raw_value})

## 4. STAGE

Сначала определяем команду каждого игрока, затем выбираем первое валидное убийство матча и связываем его с победителем. Grain итогового STAGE: одна строка на `match_id`.

In [ ]:
with SparkSessionContext("practice-case-01-stage") as ctx:
    spark = ctx
    
    with ctx.lineage(
        "practice-case-01-team-map", 
        inputs=["raw.practice_case_01_match_players"], 
        output="stage.practice_case_01_player_team_map", 
        description="Определение команды игрока по позиции в матче",
        dataset_descriptions={"stage.practice_case_01_player_team_map": "Соответствие игрока и команды в матче для определения стороны первого убийства."}
    ):
        team_map = spark.sql("""
            select match_id, account_id, case when player_slot < 128 then 'radiant' else 'dire' end as team
            from lakehouse.raw.practice_case_01_match_players
            where account_id is not null and account_id <> 4294967295
        """)
        # where account_id is not null and account_id <> 4294967295 помечу??? нужно понимать с чем связано почему
        # посмотреть по ко-во тип данных и т.д.
        team_map.writeTo("lakehouse.stage.practice_case_01_player_team_map").using("iceberg").createOrReplace()
        publish_table("stage", "practice_case_01_player_team_map", team_map, description="Команда игрока в матче для кейса первого убийства", dependencies=["raw/practice_case_01_match_players"])
        
    with ctx.lineage(
        "practice-case-01-first-blood", 
        inputs=["raw.practice_case_01_kill_events", "raw.practice_case_01_matches", "stage.practice_case_01_player_team_map"], 
        output="stage.practice_case_01_first_blood", 
        description="Определение команды, совершившей первое убийство, и победителя матча",
        dataset_descriptions={"stage.practice_case_01_first_blood": "Первое корректно зафиксированное убийство матча, команда убийцы и команда-победитель."}
    ):
        first_blood = spark.sql("""
            with ranked as (
                select match_id, time as first_blood_time, steam_account_id as attacker,
                       row_number() over (partition by match_id order by time, id) as row_num
                from lakehouse.raw.practice_case_01_kill_events
                where time >= 0 and steam_account_id is not null
            )
            select k.match_id, k.first_blood_time, t.team as first_blood_team,
                   case when m.radiant_win then 'radiant' else 'dire' end as winner_team
            from ranked k
            join lakehouse.stage.practice_case_01_player_team_map t on k.match_id = t.match_id and k.attacker = t.account_id
            join lakehouse.raw.practice_case_01_matches m on k.match_id = m.match_id
            where k.row_num = 1 and m.radiant_win is not null
        """)
        first_blood.writeTo("lakehouse.stage.practice_case_01_first_blood").using("iceberg").createOrReplace()
        publish_table("stage", "practice_case_01_first_blood", first_blood, description="Первое убийство и победившая команда по матчу", dependencies=["raw/practice_case_01_matches", "raw/practice_case_01_kill_events", "stage/practice_case_01_player_team_map"])
        
    first_blood.show(10, truncate=False)

## 5. MART

Группируем матчи по времени первого убийства. Для вопроса «после пяти минут» сравнивайте группы `00_02m` и `02_05m` с группами `05_10m` и `10m_plus`.

In [ ]:
with SparkSessionContext("practice-case-01-mart") as ctx:
    spark = ctx
    with ctx.lineage(
        "practice-case-01-first-blood-effect", 
        inputs=["stage.practice_case_01_first_blood"], 
        output="mart.practice_case_01_first_blood_effect", 
        description="Оценка связи времени первого убийства с итогом матча",
        dataset_descriptions={"mart.practice_case_01_first_blood_effect": "Сводка доли побед команды, совершившей первое убийство, по временным группам."}
    ):
        result = spark.sql("""
            select case when first_blood_time < 120 then '00_02m' when first_blood_time < 300 then '02_05m'
                        when first_blood_time < 600 then '05_10m' else '10m_plus' 
                    end as time_bucket,
                count(*) as matches,
                avg(case when first_blood_team = winner_team then 1.0 else 0.0 end) as first_blood_win_rate,
                avg(first_blood_time) as avg_first_blood_time_sec,
                avg(case when first_blood_team = winner_team then 1.0 else 0.0 end) - 0.5 as lift
            from lakehouse.stage.practice_case_01_first_blood group by 1
        """)
        result.writeTo("lakehouse.mart.practice_case_01_first_blood_effect").using("iceberg").createOrReplace()
        publish_table("mart", "practice_case_01_first_blood_effect", result, description="Влияние первого убийства на вероятность победы", dependencies=["stage/practice_case_01_first_blood"])
    result.orderBy("time_bucket").show(truncate=False)

## 6. Качество, Iceberg и вывод

Проверяем, что в итоговой таблице нет повторов матчей, вероятность находится в диапазоне от 0 до 1, а история Iceberg содержит ожидаемые снимки. `lift` показывает отличие от 50 процентов, но не доказывает, что первое убийство стало причиной победы.

In [ ]:
with SparkSessionContext("practice-case-01-quality") as spark:
    checks = {
        "stage_grain": "select count(*) - count(distinct match_id) violations from lakehouse.stage.practice_case_01_first_blood",
        "rate_range": "select count_if(first_blood_win_rate not between 0 and 1) violations from lakehouse.mart.practice_case_01_first_blood_effect",
    }
    
    for name, query in checks.items():
        violations = spark.sql(query).first()["violations"]
        print(name, "PASS" if violations == 0 else f"FAIL: {violations}")
        
    spark.sql("""
        select snapshot_id, committed_at, operation 
        from lakehouse.mart.practice_case_01_first_blood_effect.snapshots 
        order by committed_at desc limit 3
    """).show(truncate=False)
    
    spark.sql("""
        select time_bucket, matches, round(first_blood_win_rate * 100, 2) win_rate_pct, round(lift * 100, 2) lift_pp
        from lakehouse.mart.practice_case_01_first_blood_effect order by time_bucket
    """).show(truncate=False)

## 7. Аналитика MART и гипотезы

Используем MART для ответа на два вопроса: увеличивается ли вероятность победы команды, совершившей первое убийство, и меняется ли эта связь после пяти минут. Гипотеза H1: доля побед команды первого убийства выше 50 процентов. Гипотеза H2: величина связи отличается между ранними и поздними временными группами. Графики показывают размер групп и долю побед; они не доказывают причинность.

In [ ]:
import matplotlib.pyplot as plt

with SparkSessionContext("practice-case-01-analytics") as spark:
    mart = spark.table("lakehouse.mart.practice_case_01_first_blood_effect")
    analysis = (
        mart.select("time_bucket", "matches", "first_blood_win_rate", "lift", "avg_first_blood_time_sec")
            .orderBy("time_bucket")
    )
    analysis.show(truncate=False)
    plot = analysis.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(plot["time_bucket"], plot["matches"], color="#4f6d7a")
axes[0].set_title("Число матчей по времени первого убийства")
axes[0].set_ylabel("Матчи")
axes[0].tick_params(axis="x", rotation=25)
axes[1].bar(plot["time_bucket"], plot["first_blood_win_rate"] * 100, color="#c65d4b")
axes[1].axhline(50, color="#333333", linestyle="--", linewidth=1)
axes[1].set_title("Победы команды первого убийства")
axes[1].set_ylabel("Доля побед, %")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()